# Step 3. Preprocessing, EDA, and Feature Engineering

Step 2 showed what is in the data. Step 3 prepares it for modeling and studies what predicts dropout, using the right test for each feature type.

This notebook does six things.

1. Clean the data, and confirm the checks from Step 2 in code.
2. Split into train and test, so nothing downstream learns from the test set.
3. Explore each feature against dropout with the correct test for its type.
4. Engineer a small number of features from domain knowledge.
5. Select the strongest features, and reduce the numbers with PCA.
6. Build one preprocessing pipeline, fit on the training set only.

The leakage decision from Step 1 becomes code here. I drop the twelve curricular columns, so the model sees only what is known at enrollment.

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import (
    load_binary,
    CONTINUOUS, BINARY_FLAGS, NOMINAL_CODED, COUNT_ORDINAL, LEAKAGE, TARGET,
)
from src.paths import ROOT

bdf = load_binary()

# I keep enrollment-time features only. The twelve curricular columns are the
# leakage from Step 1, so I leave them out here.
FEATURES = CONTINUOUS + BINARY_FLAGS + NOMINAL_CODED + COUNT_ORDINAL
X = bdf[FEATURES].copy()
y = bdf["dropout"]
print("features kept", len(FEATURES), "| leakage columns dropped", len(LEAKAGE))

## 1. Cleaning

Step 2 found no missing values and no duplicate rows. The providers cleaned the file before release, so this confirms it in code rather than inventing work. Real extremes, like mature student ages, stay, since they are true records, not errors.

In [ ]:
print("null cells", int(X.isna().sum().sum()))
print("duplicate rows", int(X.duplicated().sum()))
print()
print("range check on true numbers")
print(X[CONTINUOUS].agg(["min", "max"]).round(1))

No null cells and no duplicate rows, as expected. Grades sit between 0 and 200, and ages run from 17 into mature entry. Nothing here needs removing. The mature ages are real students, not outliers to cut, so they stay.

## 2. Train and Test Split

I split now, before any feature work that looks at the outcome. This keeps the preparation honest. Everything that follows, the tests, the scaling, the encoding, is learned from the training set only. The test set stays back, unseen until Step 4.

The split is stratified on dropout, so both parts hold the same 39 percent rate, and the seed is fixed so the split is the same on every run.

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
print("train", Xtr.shape, "dropout rate", round(ytr.mean() * 100, 1), "percent")
print("test ", Xte.shape, "dropout rate", round(yte.mean() * 100, 1), "percent")

## 3. True Numbers Against Dropout

The six true numbers get compared between students who drop out and students who graduate. Grades and age are skewed, so the test is the Mann-Whitney U test, which compares two groups without assuming a normal shape. It asks whether the two groups differ, and returns a p-value. A small p-value means the difference is real, not chance.

The histograms show each number split by outcome, so the shape of the difference is visible.

In [ ]:
from scipy.stats import mannwhitneyu

print("Mann-Whitney U, each true number vs dropout (train)")
rows = []
for c in CONTINUOUS:
    a = Xtr.loc[ytr == 1, c]
    b = Xtr.loc[ytr == 0, c]
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    rows.append((c, round(a.median() - b.median(), 2), p))
for c, diff, p in sorted(rows, key=lambda r: r[2]):
    print(f"  {c:32} median gap {diff:8}   p={p:.1e}")

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.ravel(), CONTINUOUS):
    ax.hist(Xtr.loc[ytr == 0, c], bins=30, alpha=0.6, label="graduate", density=True)
    ax.hist(Xtr.loc[ytr == 1, c], bins=30, alpha=0.6, label="dropout", density=True)
    ax.set_title(c, fontsize=9)
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(ROOT / "reports" / "figures" / "continuous_by_dropout.png", dpi=90)
plt.show()

Age has the strongest and clearest link. Older entrants drop out more. Admission grade and previous qualification grade both matter, with lower grades tied to dropout. GDP has a weak link. Unemployment and inflation show no real difference, with p-values above 0.05, so they carry little signal on their own.

## 4. Categories Against Dropout

Categories need different tools than numbers. Three measures work together here, and it helps to know what each one does and where it comes from.

**Dropout rate per category.** The plain view. It shows the direction, which groups drop out more.

**Chi-square test.** From Karl Pearson, around 1900. It answers one yes or no question. Is the link between a category and dropout real, or could it be chance? It compares the counts seen against the counts expected if the two were unrelated, and returns a p-value. Its weakness is that it says nothing about strength. On a large sample almost any link passes as real, because the test grows with sample size.

**Cramer's V.** From Harald Cramer, 1946. It takes the chi-square number and rescales it to a 0 to 1 range, correcting for sample size and table shape. Now the result reads as strength. 0 is no link, 1 is a perfect link. Its weakness is the mirror of chi-square. It tells how strong, not whether it is beyond chance.

**Why not plain correlation.** Correlation measures a straight line link between two real numbers. On these coded categories it is wrong, since the codes are labels, not amounts. A course coded 9500 is not larger than one coded 33. This is the mistake the feature type routing exists to prevent.

**Which is better.** Neither alone. Chi-square answers is it real, Cramer's V answers how strong. Used together they rank features by a link that is both real and meaningful. On 3630 rows almost everything is significant, so Cramer's V does the real ranking for me. A rough guide for reading V, under 0.1 is negligible, 0.1 to 0.3 is weak, 0.3 to 0.5 is moderate, above 0.5 is strong.

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2, p, _, _ = chi2_contingency(table)
    n = table.sum().sum()
    r, k = table.shape
    v = np.sqrt((chi2 / n) / max(min(r, k) - 1, 1))
    return v, p

print("chi-square and Cramer's V, each category vs dropout (train)")
rows = []
for c in BINARY_FLAGS + NOMINAL_CODED:
    v, p = cramers_v(Xtr[c], ytr)
    rows.append((c, v, p))
for c, v, p in sorted(rows, key=lambda r: -r[1]):
    print(f"  {c:32} V={v:.3f}   p={p:.1e}")

Tuition fees up to date tops the ranking at 0.437, well ahead of the rest. Course, application mode, scholarship holder, and debtor follow in the moderate range. Gender is moderate too. Nationality, international, and educational special needs sit near zero with p-values above 0.05, so they carry almost no signal.

The next check looks at links between categories, to find columns that repeat the same information.

In [ ]:
cats = BINARY_FLAGS + NOMINAL_CODED
M = pd.DataFrame(np.eye(len(cats)), index=cats, columns=cats)
for i, a in enumerate(cats):
    for j, b in enumerate(cats):
        if i < j:
            v, _ = cramers_v(Xtr[a], Xtr[b])
            M.iloc[i, j] = M.iloc[j, i] = v

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(M.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=90, fontsize=7)
ax.set_yticks(range(len(cats))); ax.set_yticklabels(cats, fontsize=7)
fig.colorbar(im); fig.tight_layout()
fig.savefig(ROOT / "reports" / "figures" / "cramers_v_matrix.png", dpi=90)
plt.show()

pairs = [(cats[i], cats[j], M.iloc[i, j]) for i in range(len(cats)) for j in range(len(cats)) if i < j]
print("strongest category to category links")
for a, b, v in sorted(pairs, key=lambda r: -r[2])[:4]:
    print(f"  {a} + {b}   V={v:.3f}")

Two pairs are perfectly linked. International repeats Nationality, since one is derived from the other. Daytime or evening attendance is fixed by Course. Mother's and father's occupation overlap strongly. These are redundant columns, and a model does not need both sides of a perfect pair.

One more check guards the linear model. Multicollinearity, where true numbers move together, makes a linear model unstable. The variance inflation factor, VIF, measures it. A VIF near 1 is clean, above 5 is a concern.

In [ ]:
corr = Xtr[CONTINUOUS].corr().values
vif = np.diag(np.linalg.inv(corr))
print("VIF on the six true numbers (train)")
for c, v in sorted(zip(CONTINUOUS, vif), key=lambda r: -r[1]):
    print(f"  {c:32} VIF={v:.2f}")

Every VIF is under 2, with the highest near 1.5. The true numbers are close to independent, so the linear model is safe on this front, and nothing is dropped for multicollinearity.

## 5. The Tuition Status Check

Step 2 flagged tuition fees up to date as a near-outcome signal. Students not up to date drop out 94 percent of the time, and it tops the Cramer's V ranking. A student who has stopped paying is often a student already leaving, so the flag sits close to the outcome.

Here a quick logistic regression trains twice, once with the flag and once without, compared on the held-out test set. This measures how much the model leans on it.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, roc_auc_score

def quick_lr(cols):
    pre = ColumnTransformer([
        ("num", StandardScaler(), [c for c in CONTINUOUS + COUNT_ORDINAL if c in cols]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in NOMINAL_CODED if c in cols]),
        ("flag", "passthrough", [c for c in BINARY_FLAGS if c in cols]),
    ])
    model = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))])
    model.fit(Xtr[cols], ytr)
    pred = model.predict(Xte[cols])
    proba = model.predict_proba(Xte[cols])[:, 1]
    return recall_score(yte, pred), roc_auc_score(yte, proba)

with_t = quick_lr(FEATURES)
without_t = quick_lr([c for c in FEATURES if c != "Tuition fees up to date"])
print(f"with tuition     recall={with_t[0]:.3f}   roc_auc={with_t[1]:.3f}")
print(f"without tuition  recall={without_t[0]:.3f}   roc_auc={without_t[1]:.3f}")

Removing the flag lowers ROC AUC from 0.855 to 0.809 and recall from 0.771 to 0.750. So it is a real contributor, not a lone driver. The model still works well without it. I keep the flag for now, and record its near-outcome nature as a known risk. A stricter stance would drop it, at a modest cost, which is the same early-warning trade the project already accepts.

## 6. Feature Engineering

One feature comes from domain knowledge. Three flags point at money pressure, being a debtor, not being up to date on tuition, and not holding a scholarship. Each is a sign of financial strain, and strain is tied to dropout. I add them into a socioeconomic pressure score from 0 to 3, a single feature that captures how many strain signals a student carries.

It is built after the split, using a fixed rule, so it does not learn anything from the data and cannot leak.

In [ ]:
def add_pressure(frame):
    frame = frame.copy()
    frame["socioeconomic pressure"] = (
        frame["Debtor"]
        + (1 - frame["Tuition fees up to date"])
        + (1 - frame["Scholarship holder"])
    )
    return frame

Xtr = add_pressure(Xtr)
Xte = add_pressure(Xte)
print("socioeconomic pressure, train counts")
print(Xtr["socioeconomic pressure"].value_counts().sort_index())

## 7. Feature Selection

Filter selection ranks features by how much they tell about dropout, before any model is trained. The measure is mutual information, which captures any kind of dependence, not just a straight line, and works across mixed feature types. The categories are marked as discrete so their codes are read as labels, not amounts.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

cols = FEATURES + ["socioeconomic pressure"]
discrete = [c in (BINARY_FLAGS + NOMINAL_CODED) for c in cols]
mi = mutual_info_classif(Xtr[cols], ytr, discrete_features=discrete, random_state=42)
print("mutual information, top features (train)")
for c, m in sorted(zip(cols, mi), key=lambda r: -r[1])[:8]:
    print(f"  {c:32} MI={m:.4f}")

The engineered pressure score ranks first, ahead of tuition status alone, which shows the composite adds value. Course, age, scholarship, and application mode follow. The weakest features, nationality, international, and educational special needs, match the near-zero Cramer's V from earlier, so the two methods agree on what to drop.

## 8. Dimensionality Reduction with PCA

PCA compresses several correlated numbers into fewer combined ones. It fits true numbers only, since it relies on distance and order, which the coded categories do not have. Here it runs on the six true numbers to see how much they can be compressed.

In [ ]:
from sklearn.decomposition import PCA

Z = StandardScaler().fit_transform(Xtr[CONTINUOUS])
pca = PCA().fit(Z)
cum = np.cumsum(pca.explained_variance_ratio_)
print("variance explained per component", [round(x, 3) for x in pca.explained_variance_ratio_])
print("cumulative                     ", [round(x, 3) for x in cum])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, 7), cum, marker="o")
ax.set_xlabel("components"); ax.set_ylabel("cumulative variance"); ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(ROOT / "reports" / "figures" / "pca_scree.png", dpi=90)
plt.show()

No component dominates. The first holds only 27 percent, and it takes four of the six to reach 83 percent. This matches the low VIF from earlier. The true numbers are close to independent, so PCA buys little here. That is an honest result, not a failure. PCA stays as a documented option for the linear model, and the trees do not need it.

## 9. The Preprocessing Pipeline

One pipeline holds all preparation. It scales the true numbers, one-hot encodes the coded categories, and passes the flags through unchanged. It is fit on the training set only, so the test set stays untouched until Step 4.

Two artifacts are saved. The fitted pipeline goes to models, and the train and test frames go to data/processed. Step 4 loads these and never re-splits or re-fits preparation.

In [ ]:
import joblib

NUM = CONTINUOUS + COUNT_ORDINAL + ["socioeconomic pressure"]
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), NOMINAL_CODED),
    ("flag", "passthrough", BINARY_FLAGS),
])
preprocessor.fit(Xtr)
print("prepared training shape", preprocessor.transform(Xtr).shape)

(ROOT / "models").mkdir(exist_ok=True)
(ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, ROOT / "models" / "preprocessor.joblib")
Xtr.assign(dropout=ytr.values).to_csv(ROOT / "data" / "processed" / "train.csv", index=False)
Xte.assign(dropout=yte.values).to_csv(ROOT / "data" / "processed" / "test.csv", index=False)
print("saved preprocessor.joblib and train.csv and test.csv")

## What Step 3 Settles

The base for Step 4 is ready.

1. Only enrollment-time features remain, since the twelve curricular columns are dropped as leakage.
2. The data is split, stratified on dropout, with the test set held back.
3. Age and grades are the strongest numbers, and tuition status, course, scholarship, and debtor are the strongest categories.
4. A socioeconomic pressure feature was added that ranks first on mutual information.
5. Redundant columns and multicollinearity were checked, and PCA was tested with an honest verdict.
6. One preprocessing pipeline is fit on train only and saved, with the split, for Step 4.

Step 4 loads the saved split and pipeline, trains several models, and compares them.